In [1]:
# =========================================================
# XGBOOST + SARIMAX + SEASONAL NAIVE FORECAST COMPARISON
# =========================================================

import sys
sys.path.append("../src")

import importlib
import sarimax_utils
importlib.reload(sarimax_utils)
from sarimax_utils import compare_sarimax_models

import pandas as pd
from data_utils import split_data
from config import get_country_settings, get_exog_specs
from xgboost_utils import (
    get_xgboost_feature_sets,
    fit_predict_xgboost_models,
    seasonal_naive_forecast,
    build_sarimax_forecasts,
    build_comparison_plot_df,
    get_xgb_gain_importance,
    get_xgb_shap_importance,
    make_pretty_feature_names,
    build_shap_comparison_table,
    save_shap_outputs
)


# Load processed data
df = pd.read_csv("../data/processed_data.csv", parse_dates=True, index_col=0)

df = df.sort_index()
df = df.asfreq("D")
df = df.dropna()

In [2]:
train_start_year = 2010
forecast_year = 2025

train_data, val_data, development_data, forecast_data = split_data(
    df,
    train_start_year=train_start_year,
    forecast_year=forecast_year
)

In [3]:
country = "Norway"

code, target_col, lag_prefix = get_country_settings(country)
specs = get_exog_specs(country)

full_exog_vars = specs["Full"]
restricted_exog_vars = specs["Restricted"]

In [4]:
# =========================================================
# 1. XGBOOST FEATURE SETS
# =========================================================

feature_sets = get_xgboost_feature_sets(
    df=df,
    development_data=development_data,
    country=country,
    lag_prefix=lag_prefix,
    full_exog_vars=full_exog_vars,
    restricted_exog_vars=restricted_exog_vars
)

for model_name, features in feature_sets.items():
    print("=" * 70)
    print(model_name)
    print(features)

XGBoost Full
['HDD_Norway', 'Extreme_Cold_NO', 'HDD_Extreme_NO', 'CDD_Norway', 'Extreme_Warm_NO', 'CDD_Extreme_NO', 'lag1_log_no', 'Holiday_Norway', 'day_Monday', 'day_Tuesday', 'day_Wednesday', 'day_Thursday', 'day_Saturday', 'day_Sunday', 'month_February', 'month_March', 'month_April', 'month_May', 'month_June', 'month_July', 'month_August', 'month_September', 'month_October', 'month_November', 'month_December', 'lag2_log_no', 'lag3_log_no', 'lag4_log_no', 'lag5_log_no', 'lag6_log_no', 'lag7_log_no']
XGBoost Restricted
['HDD_Norway', 'Extreme_Cold_NO', 'HDD_Extreme_NO', 'lag1_log_no', 'Holiday_Norway', 'day_Monday', 'day_Tuesday', 'day_Wednesday', 'day_Thursday', 'day_Saturday', 'day_Sunday', 'month_February', 'month_March', 'month_April', 'month_May', 'month_June', 'month_July', 'month_August', 'month_September', 'month_October', 'month_November', 'month_December', 'lag2_log_no', 'lag3_log_no', 'lag4_log_no', 'lag5_log_no', 'lag6_log_no', 'lag7_log_no']
XGBoost No Weather
['Holiday_

In [5]:
# =========================================================
# 2. FIT AND FORECAST XGBOOST MODELS
# =========================================================

xgb_predictions_log, fitted_xgb_models, xgb_test_targets = fit_predict_xgboost_models(
    development_data=development_data,
    forecast_data=forecast_data,
    target_col=target_col,
    feature_sets=feature_sets
)

In [6]:
from sarimax_utils import compare_sarimax_models

model_specs = {
    "SARIMAX Full": {
        "Specification": "Full",
        "exog_vars": full_exog_vars,
        "order": (1, 0, 0),
        "seasonal_order": (0, 1, 1, 7)
    },
    "SARIMAX Restricted": {
        "Specification": "Restricted",
        "exog_vars": restricted_exog_vars,
        "order": (0, 0, 1),
        "seasonal_order": (0, 1, 1, 7)
    }
}

sarimax_comparison_df, fitted_sarimax_models = compare_sarimax_models(
    model_specs=model_specs,
    development_data=development_data,
    forecast_data=forecast_data,
    target_col=target_col,
    country=country
)

Maximum Likelihood optimization failed to converge. Check mle_retvals
Maximum Likelihood optimization failed to converge. Check mle_retvals


In [7]:
# =========================================================
# 3. BUILD SARIMAX FORECASTS
# =========================================================

sarimax_predictions_log = build_sarimax_forecasts(
    forecast_data=forecast_data,
    target_col=target_col,
    model_specs=model_specs,
    fitted_sarimax_models=fitted_sarimax_models
)

In [8]:
# =========================================================
# 4. SEASONAL NAIVE FORECAST
# =========================================================

naive_pred_log = seasonal_naive_forecast(
    df=df,
    target_col=target_col,
    forecast_data=forecast_data,
    seasonal_lag=7
)

In [9]:
# =========================================================
# 5. COMBINE ALL FORECASTS
# =========================================================

predictions_log = {}

predictions_log.update(sarimax_predictions_log)
predictions_log.update(xgb_predictions_log)

predictions_log["Seasonal Naïve"] = naive_pred_log

y_test_log = forecast_data[target_col].copy()

comparison_plot_df = build_comparison_plot_df(
    y_test_log=y_test_log,
    predictions_log=predictions_log
)

print(comparison_plot_df.shape)
comparison_plot_df.head()

(365, 7)


,Actual,SARIMAX Full,SARIMAX Restricted,XGBoost Full,XGBoost Restricted,XGBoost No Weather,Seasonal Naïve
Load Date,,,,,,,
2025-01-01,13.056935,13.040005,13.036235,13.050311,13.048638,13.035567,12.861907
2025-01-02,13.135830,13.140008,13.127316,13.092100,13.089729,13.065387,12.880093
2025-01-03,13.147364,13.175441,13.167610,13.133213,13.131088,13.141817,12.946869
2025-01-04,13.131710,13.142813,13.131341,13.132871,13.118542,13.095358,12.925609
2025-01-05,13.143628,13.150872,13.139012,13.122243,13.115834,13.113643,12.920952


XGBOOST FEATURE IMPORTANCE

In [10]:
# Recreate train/test matrices for SHAP
xgb_data = {}

for model_name, features in feature_sets.items():
    train = development_data[[target_col] + features].dropna().copy()
    test = forecast_data[[target_col] + features].dropna().copy()

    xgb_data[model_name] = {
        "X_train": train[features],
        "X_test": test[features]
    }

In [11]:
gain_importance_dict = {}
shap_values_dict = {}
shap_importance_dict = {}

for model_name, model in fitted_xgb_models.items():
    X_train = xgb_data[model_name]["X_train"]
    X_test = xgb_data[model_name]["X_test"]

    gain_importance_dict[model_name] = get_xgb_gain_importance(
        model=model,
        feature_names=X_train.columns
    )

    shap_values, shap_importance = get_xgb_shap_importance(
        model=model,
        X_test=X_test
    )

    shap_values_dict[model_name] = shap_values
    shap_importance_dict[model_name] = shap_importance

    print(f"\n{model_name} - Gain importance")
    print(gain_importance_dict[model_name].head(15))

    print(f"\n{model_name} - SHAP importance")
    print(shap_importance_dict[model_name].head(15))


XGBoost Full - Gain importance
            Feature  Importance
0       lag1_log_no    0.607919
1        HDD_Norway    0.159395
2        day_Monday    0.045600
3       lag7_log_no    0.044892
4      day_Saturday    0.023668
5   Extreme_Cold_NO    0.013404
6        day_Sunday    0.012466
7       lag2_log_no    0.011858
8    Holiday_Norway    0.011766
9       month_April    0.009445
10    month_October    0.008542
11      month_March    0.005835
12   HDD_Extreme_NO    0.005686
13   month_November    0.005273
14        month_May    0.004970

XGBoost Full - SHAP importance
           Feature  MeanAbsSHAP
0      lag1_log_no     0.101931
1       HDD_Norway     0.048117
2     day_Saturday     0.010121
3      lag7_log_no     0.009639
4       day_Monday     0.008958
5       day_Sunday     0.004609
6   Holiday_Norway     0.002583
7      lag6_log_no     0.002163
8      month_April     0.001755
9      lag2_log_no     0.001690
10     lag4_log_no     0.001570
11     month_March     0.001493
12     l

In [12]:
pretty_feature_names = make_pretty_feature_names(
    country=country,
    code=code,
    lag_prefix=lag_prefix
)

shap_compare = build_shap_comparison_table(
    shap_importance_dict=shap_importance_dict,
    pretty_feature_names=pretty_feature_names
)

shap_compare.head(15)

,Feature,MeanAbsSHAP_Full,MeanAbsSHAP_Restricted,MeanAbsSHAP_NoWeather,PrettyFeature,TotalImportance
0,lag1_log_no,0.101931,0.103733,0.129997,Lag 1,0.335661
1,HDD_Norway,0.048117,0.045080,0.000000,HDD,0.093197
2,day_Saturday,0.010121,0.010143,0.010381,Saturday,0.030645
3,day_Monday,0.008958,0.009215,0.012470,Monday,0.030643
4,lag7_log_no,0.009639,0.007221,0.009306,Lag 7,0.026166
5,day_Sunday,0.004609,0.005141,0.002729,Sunday,0.012479
6,lag2_log_no,0.001690,0.004973,0.005068,Lag 2,0.011730
7,lag3_log_no,0.001188,0.000864,0.009489,Lag 3,0.011541
8,lag6_log_no,0.002163,0.002251,0.005284,Lag 6,0.009698
9,Holiday_Norway,0.002583,0.002594,0.002169,Holiday,0.007346


In [13]:
output_path = f"../outputs/tables/{country}_comparison_plot_df_{forecast_year}.csv"

comparison_plot_df.to_csv(output_path)

print(f"Saved: {output_path}")

Saved: ../outputs/tables/Norway_comparison_plot_df_2025.csv


In [14]:
excel_path, csv_path = save_shap_outputs(
    country=country,
    forecast_year=forecast_year,
    gain_importance_dict=gain_importance_dict,
    shap_importance_dict=shap_importance_dict,
    shap_compare=shap_compare,
    output_dir="../outputs/tables"
)

print(f"Saved Excel: {excel_path}")
print(f"Saved CSV: {csv_path}")

Saved Excel: ../outputs/tables\Norway_xgboost_shap_importance_2025.xlsx
Saved CSV: ../outputs/tables\shap_compare_Norway_2025.csv
